In [ ]:
!pip install langchain chromadb sentence-transformers PyPDF2 pydantic transformers -q

: 

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
from typing import List, Dict, Tuple
from collections import defaultdict

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import Document
from langchain.document_loaders import PyPDFLoader, TextLoader

print("✓ All imports successful")

In [ ]:
sample_docs_text = {
    "machine_learning.txt": """Machine Learning Fundamentals

Machine learning is a subset of artificial intelligence that enables computers to learn from data without being explicitly programmed.

Types of Machine Learning:

1. Supervised Learning: The model learns from labeled training data.
   - Classification: Predicting discrete categories (spam detection, image classification)
   - Regression: Predicting continuous values (house prices, temperature)

2. Unsupervised Learning: The model discovers patterns in unlabeled data.
   - Clustering: Grouping similar data points (customer segmentation)
   - Dimensionality Reduction: Reducing features while preserving information

3. Reinforcement Learning: The model learns by interacting with environment and receiving rewards.
   - Used in game playing, robotics, autonomous systems
   - Examples: AlphaGo, self-driving cars

Common algorithms: Decision Trees, Random Forests, SVM, Neural Networks, K-Means clustering.
The key to successful ML is quality data, feature engineering, and proper model evaluation.""",

    "deep_learning.txt": """Deep Learning Concepts

Deep learning uses artificial neural networks with multiple layers.

Key Components:
- Neurons: Basic computational units processing inputs and outputs
- Layers: Collections of neurons working together
- Activation Functions: ReLU, Sigmoid, Tanh introduce non-linearity
- Backpropagation: Algorithm for training networks by computing gradients

Types of Neural Networks:
- CNN (Convolutional Neural Networks): For image processing
- RNN (Recurrent Neural Networks): For sequential data and time series
- Transformers: Attention-based, state-of-the-art for NLP
- Autoencoders: Unsupervised learning and dimensionality reduction

Applications:
- Computer Vision: Object detection, classification, segmentation
- NLP: Machine translation, sentiment analysis, language modeling
- Speech Recognition: Converting audio to text
- Recommendation Systems: Personalized suggestions

Deep learning achieves superhuman performance but requires significant resources and data.""",

    "nlp_guide.txt": """Natural Language Processing Guide

NLP enables computers to understand, interpret, and generate human language.

Core NLP Tasks:
1. Tokenization: Breaking text into words, phrases, sentences
2. Part-of-Speech Tagging: Identifying nouns, verbs, adjectives
3. Named Entity Recognition: Finding people, organizations, locations
4. Sentiment Analysis: Determining emotional tone of text
5. Machine Translation: Converting text between languages
6. Question Answering: Extracting answers from text based on questions
7. Text Summarization: Condensing text while preserving key information

Popular NLP Models:
- BERT: Bidirectional Encoder Representations from Transformers
- GPT: Generative Pre-trained Transformer models
- Word2Vec and GloVe: Word embedding techniques
- T5: Text-to-Text Transfer Transformer

NLP Preprocessing:
- Lowercasing: Converting to lowercase
- Removing punctuation and special characters
- Removing stopwords: Common words without meaning
- Lemmatization: Reducing words to base form
- Stemming: Reducing words to root form

Modern NLP uses Transformers and pre-trained models."""
}

docs_dir = Path("sample_docs")
docs_dir.mkdir(exist_ok=True)

for filename, content in sample_docs_text.items():
    filepath = docs_dir / filename
    with open(filepath, "w") as f:
        f.write(content)
    print(f"✓ Created {filename}")

print(f"\nCreated {len(sample_docs_text)} sample documents")

In [ ]:
def load_documents(docs_dir: str) -> List[Document]:
    documents = []
    docs_path = Path(docs_dir)
    
    for txt_file in docs_path.glob("*.txt"):
        loader = TextLoader(str(txt_file), encoding='utf-8')
        documents.extend(loader.load())
        print(f"✓ Loaded {txt_file.name}")
    
    for pdf_file in docs_path.glob("*.pdf"):
        loader = PyPDFLoader(str(pdf_file))
        documents.extend(loader.load())
        print(f"✓ Loaded {pdf_file.name}")
    
    return documents

documents = load_documents("sample_docs")
print(f"\n✓ Total documents: {len(documents)}")
print(f"✓ Total characters: {sum(len(doc.page_content) for doc in documents):,}")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"✓ Total chunks created: {len(chunks)}")
chunk_sizes = [len(chunk.page_content) for chunk in chunks]
print(f"\nChunk statistics:")
print(f"  Average size: {np.mean(chunk_sizes):.0f} characters")
print(f"  Min size: {np.min(chunk_sizes)} characters")
print(f"  Max size: {np.max(chunk_sizes)} characters")

In [ ]:
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
print("✓ Embeddings model loaded")

persist_directory = "./chroma_db"

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_documents"
)

print(f"✓ ChromaDB created with {len(chunks)} embeddings")
print(f"✓ Persisted to: {persist_directory}")

test_embedding = embeddings.embed_query("test")
print(f"✓ Embedding dimension: {len(test_embedding)}")

In [ ]:
class RAGRetriever:
    def __init__(self, vectorstore, top_k: int = 3):
        self.vectorstore = vectorstore
        self.top_k = top_k
    
    def retrieve(self, query: str) -> Tuple[List[str], List[float]]:
        results = self.vectorstore.similarity_search_with_scores(query, k=self.top_k)
        contexts = [doc.page_content for doc, _ in results]
        scores = [score for _, score in results]
        return contexts, scores

retriever = RAGRetriever(vectorstore, top_k=3)
print("✓ RAG Retriever initialized")

In [ ]:
test_queries = [
    "What are the types of machine learning?",
    "Explain neural networks",
    "What is NLP?"
]

print("="*70)
print("RETRIEVAL TEST")
print("="*70)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 70)
    contexts, scores = retriever.retrieve(query)
    
    for i, (context, score) in enumerate(zip(contexts, scores)):
        print(f"\n[Result {i+1}] Similarity: {score:.4f}")
        print(context[:200] + "..." if len(context) > 200 else context)